## Ran Uram 206661886
## Shahar Lankry 322659137
## Daniel Geron 212515522
-------------------------
# Feature Extraction and Report

## Imports

In [170]:
import warnings
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re
from typing import Tuple
from tqdm import tqdm
from openpyxl import load_workbook
# import anthropic

## Paths and Configuration

In [171]:
normalised_images_folder =r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalDataNormalized"
segment_results_folder  =r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/Data_normalized"
feature_tables_location =r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/tables"
COLUMN_TOKEN = "_COLUMNS"

os.makedirs(feature_tables_location, exist_ok=True)

visualization_folder = r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/visualization_results"
os.makedirs(visualization_folder, exist_ok=True)
for feature in ['baseline','slant','stroke_thickness','right_margin','left_margin','word_aspect_ratio','baseline_slope','top_margin','bottom_margin','column_spacing']:
    os.makedirs(os.path.join(visualization_folder, feature), exist_ok=True)

## FEATURE EXTRACTION

<table dir="rtl" style="border-collapse: collapse; width: 100%; text-align: right; border: 1px solid black;">
    <thead>
        <tr>
            <th style="border: 1px solid black; padding: 10px;">ערכי הקיצון והאמצע (סקאלה 0-1)</th>
            <th style="border: 1px solid black; padding: 10px;">מה זה אומר? (תיאור)</th>
            <th style="border: 1px solid black; padding: 10px;">שם הפיצ'ר</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = שמאלית חזקה (נגד הכיוון)<br>0.5 = אנכי לגמרי (90 מעלות, ישר)<br>1.0 = ימנית חזקה (שוכב קדימה)</td>
            <td style="border: 1px solid black; padding: 10px;">זווית הכתיבה ביחס לאנך</td>
            <td style="border: 1px solid black; padding: 10px;">נטייה (Slant)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = דקיק (קו נימי, עדין מאוד)<br>0.5 = בינוני (עובי סטנדרטי)<br>1.0 = עבה מאוד (קו "בצקי", מרוח)</td>
            <td style="border: 1px solid black; padding: 10px;">עובי הקו ביחס לגודל האות ("לחץ")</td>
            <td style="border: 1px solid black; padding: 10px;">עובי קו (Stroke Thickness)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = חותך/יורד (הכתיבה יורדת מתחת לקו)<br>0.5 = על הקו בדיוק (התאמה מושלמת)<br>1.0 = מרחף גבוה (הכתיבה מנותקת מהקו כלפי מעלה)</td>
            <td style="border: 1px solid black; padding: 10px;">מיקום האות ביחס לפס המודפס</td>
            <td style="border: 1px solid black; padding: 10px;">היצמדות לשורה (Baseline)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = צמוד לקצה הימני (0% עד 5% מרוחב הדף)<br>0.5 = מרחק מאוזן מהקצה (10% עד 15%)<br>1.0 = מרחק גדול מהקצה הימני (מעל 30%)</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק מהפיקסל השחור הראשון בשורה לקצה הימני של המסגרת</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>מדידת שוליים- ימין (Margin Right)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = צמוד לקצה השמאלי (0% עד 5% מרוחב הדף)<br>0.5 = מרחק מאוזן מהקצה (10% עד 15%)<br>1.0 = מרחק גדול מהקצה השמאלי (מעל 30%)</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק מהפיקסל האחרון לקצה השמאלי של המסגרת</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>מדידת שוליים- שמאל (Margin Left)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">-1 = זוהתה מילה אחת בלבד (לא ניתן למדוד)<br>0.0 = מילים דבוקות (אין רווח)<br>0.5 = רווח תקין בין מילים<br>1.0 = רווחים גדולים מאוד בין מילים</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק החציוני בין מילים שזוהו בשורה</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>רווח בין מילים (Word Spacing)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = כתב זעיר- תופס שטח מינימלי מהשורה<br>0.5 = כתב בינוני ותקני- פרופורציה הגיונית<br>1.0 = כתב ענק- משתלט על מרחב השורה</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק האנכי של האותיות, מחושב כחציון גובה התיבות התוחמות ומנורמל לגובה הדף</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>גודל הכתב האבסולוטי (Letter Size)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = כתב זוויתי- קווים חדים ושפיצים<br>0.5 = שילוב מאוזן- עקומות מתונות וזורמות<br>1.0 = כתב עגול- צורות עגולות, רכות ומלאות</td>
            <td style="border: 1px solid black; padding: 10px;">בדיקת ה"שפיציות" של הכתב דרך חישוב מדד המעגליות של החללים הלבנים בתוך האותיות</td>
            <td style="border: 1px solid black; padding: 10px;">מעגליות מול זוויתיות (Roundness vs. Angularity)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = שורה נופלת (שיפוע רגרסיה חיובי)<br>0.5 = שורה ישרה (קו אופקי ושטוח)<br>1.0 = שורה מטפסת (שיפוע רגרסיה שלילי)</td>
            <td style="border: 1px solid black; padding: 10px;">כיוון הזרימה של השורה, מחושב באמצעות רגרסיה לינארית על מרכזי הכובד של המילים</td>
            <td style="border: 1px solid black; padding: 10px;">שיפוע קו הבסיס (Baseline Slope)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = מילים צרות ומכווצות- דחוסות כלפי פנים<br>0.5 = פרופורציה מאוזנת- יחס רוחב-גובה תקין<br>1.0 = מילים רחבות ומתוחות- מרוחות על הדף</td>
            <td style="border: 1px solid black; padding: 10px;">בדיקת "מתיחת" המילה בחלל, מחושב כחציון היחס בין רוחב התיבה התוחמת לגובהה</td>
            <td style="border: 1px solid black; padding: 10px;">יחס רוחב-גובה של המילים (Word Aspect Ratio)</td>
        </tr>
    </tbody>
</table>

### Slant

In [172]:
def measure_slant_by_shear(binary_img: np.ndarray) -> Tuple[float, float]:
    # Calculates the slant by applying affine shear transformations to the image.
    # Logic: We shear the image at different angles. The angle that maximizes the 
    # variance of the vertical projection is the one that makes the text most upright.
    
    img_height, img_width = binary_img.shape
    
    # Test angles from -30 to +30 degrees
    angles_to_test = np.linspace(-30, 30, 61)
    max_variance = 0
    optimal_angle = 0

    for angle in angles_to_test:
        angle_rad = np.radians(angle)
        shear_factor = np.tan(angle_rad)
        
        # Define the Affine Transformation Matrix for shearing
        M = np.array([[1, shear_factor, 0],
                      [0, 1, 0]], dtype=np.float32)
        
        # Apply the shear transformation
        sheared = cv2.warpAffine(binary_img, M, (img_width + abs(int(shear_factor * img_height)), img_height),
                                flags=cv2.INTER_LINEAR, borderValue=255)
        
        # Calculate vertical projection (sum of black pixels per column)
        projection = np.sum(sheared == 0, axis=0)
        
        # Calculate variance - higher variance means sharper peaks/valleys, indicating upright text
        variance = np.var(projection)

        if variance > max_variance:
            max_variance = variance
            optimal_angle = angle

    # Normalize the result to 0.0 - 1.0 range
    # -30 deg = 0.0, 0 deg = 0.5, +30 deg = 1.0
    slant = (optimal_angle + 30) / 60
    slant = np.clip(slant, 0.0, 1.0)
    
    return slant, optimal_angle

In [173]:
def measure_slant_by_moments(binary_img: np.ndarray) -> float:
    # Calculates slant by finding contours of individual letters and fitting ellipses.
    # The average angle of these ellipses represents the handwriting slant.
    
    # Invert image to find contours (OpenCV expects white object on black background)
    inverted = 255 - binary_img
    contours, _ = cv2.findContours(inverted, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return 0.5

    angles = []
    img_area = binary_img.shape[0] * binary_img.shape[1]

    for contour in contours:
        area = cv2.contourArea(contour)
        
        # Filter noise: Ignore contours that are too small or too large
        if area < img_area * 0.0001 or area > img_area * 0.15:
            continue
        if len(contour) < 5:
            continue

        try:
            # Fit an ellipse to the contour to determine orientation
            (x, y), (MA, ma), angle = cv2.fitEllipse(contour)
            
            # Adjust angle to be relative to vertical axis
            if angle > 135:
                angle = angle - 180
            elif angle > 45:
                angle = 90 - angle
            angles.append(angle)
        except:
            continue

    if not angles:
        return 0.5

    # Remove statistical outliers using Interquartile Range (IQR) to get a robust mean
    angles = np.array(angles)
    q1, q3 = np.percentile(angles, [25, 75])
    iqr = q3 - q1
    if iqr > 0:
        mask = (angles >= q1 - 1.5*iqr) & (angles <= q3 + 1.5*iqr)
        angles = angles[mask]

    if len(angles) == 0:
        return 0.5

    mean_angle = np.mean(angles)
    
    # Normalize to 0.0 - 1.0 range based on -20 to +20 degrees limits
    mean_angle = np.clip(mean_angle, -20, 20)
    slant = (mean_angle + 20) / 40
    return slant

In [174]:
def extract_slant(image_path: str) -> Tuple[float, dict]:
    # Main function to process a single image.
    # Combines Shear method (global) and Moments method (local) for better accuracy.
    
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.5, {'error': 'Failed to read'}

    # Ensure binary image (Thresholding) if not already
    if len(np.unique(img)) > 2:
        _, img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    # Run both algorithms
    slant_shear, optimal_angle = measure_slant_by_shear(img)
    slant_moments = measure_slant_by_moments(img)

    # Weighted Average: Giving more weight (0.7) to the Shear method as it is generally more robust for lines
    slant = 0.7 * slant_shear + 0.3 * slant_moments
    slant = np.clip(slant, 0.0, 1.0)
    debug_info = {
        'slant_shear': slant_shear,
        'slant_moments': slant_moments,
        'optimal_angle': optimal_angle
    }
    return slant, debug_info

### Stroke Thickness

In [175]:
def remove_printed_text_and_lines(img: np.ndarray) -> np.ndarray:
    # Preprocesses image to remove horizontal ruled lines and noise, isolating handwriting
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Create binary inverse (white text on black background)
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    height, width = img.shape[:2]

    # Detect and remove horizontal lines using Hough Transform
    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi/180, threshold=100, minLineLength=width*0.3, maxLineGap=10)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Draw thick lines over detected ruled lines to mask them
            cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 8)

    # Subtract lines from the binary image
    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))
    
    # Filter out noise and irrelevant regions using Connected Components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        # Define regions to ignore (header, footer) and noise threshold
        is_bottom_region = (y + h) > height * 0.85
        is_top_region = y < height * 0.1
        is_too_small = area < 15

        # Keep only valid handwriting components
        if not (is_bottom_region or is_top_region or is_too_small):
            handwriting_mask[labels == i] = 255

    # Invert back to normal (black text on white background)
    result = cv2.bitwise_not(handwriting_mask)
    return result

In [176]:
def calculate_stroke_thickness_pure(img: np.ndarray) -> float:
    # Calculates average stroke thickness using Distance Transform
    handwriting_only = remove_printed_text_and_lines(img)

    if len(handwriting_only.shape) == 3:
        gray = cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
    else:
        gray = handwriting_only

    # Binary thresholding
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    inverted = cv2.bitwise_not(binary)

    # Distance Transform: calculates distance of each pixel to the nearest zero pixel (background)
    # This effectively measures the "skeleton" width at every point
    dist_transform = cv2.distanceTransform(inverted, cv2.DIST_L2, 5)
    text_distances = dist_transform[dist_transform > 0]

    # Calculate ratio of handwriting pixels to total area
    total_pixels = inverted.shape[0] * inverted.shape[1]
    handwriting_ratio = len(text_distances) / total_pixels if total_pixels > 0 else 0

    # Safety check: If page is empty or has too little content, try raw image or return 0
    MINIMUM_CONTENT_THRESHOLD = 0.003 
    if len(text_distances) == 0 or handwriting_ratio < MINIMUM_CONTENT_THRESHOLD:
        if len(img.shape) == 3:
            gray_raw = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray_raw = img

        _, binary_raw = cv2.threshold(gray_raw, 127, 255, cv2.THRESH_BINARY)
        inverted_raw = cv2.bitwise_not(binary_raw)

        dist_transform_raw = cv2.distanceTransform(inverted_raw, cv2.DIST_L2, 5)
        text_distances = dist_transform_raw[dist_transform_raw > 0]

        if len(text_distances) == 0:
            return 0.0
    
    # Mean distance is the radius, so multiply by 2 for diameter (thickness)
    avg_thickness = np.mean(text_distances) * 2

    return avg_thickness


### Baseline 

In [177]:
def extract_file_id(filename: str) -> int:
    # Helper function to extract the first number from a filename for natural sorting
    match = re.search(r'(\d+)', filename)
    if match:
        return int(match.group(1))
    return -1

In [178]:
def calculate_position_score(img: np.ndarray) -> float:
    # core logic: Calculates vertical position relative to the baseline.
    # Returns a score between 0.0 (Cutting the line) and 1.0 (Floating above).
    copy_img=img.copy()
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Otsu's thresholding to create binary image
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    img_h, img_w = thresh.shape

    # Define a wide horizontal kernel to detect only the ruled lines (ignoring text)
    min_line_width = int(img_w * 0.3)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
    
    # Morphological opening to isolate horizontal lines
    detected_lines_map = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    
    cnts_lines, _ = cv2.findContours(detected_lines_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not cnts_lines:
        return None,copy_img

    valid_lines = []
    for c in cnts_lines:
        x, y, w, h = cv2.boundingRect(c)
        # Filter out lines that are too close to the top/bottom borders (noise)
        if y > 10 and y < img_h - 10:
            valid_lines.append(c)
            
    if not valid_lines:
        valid_lines = [max(cnts_lines, key=cv2.contourArea)]

    # Subtract the detected lines from original binary to get only the handwriting
    text_only = cv2.subtract(thresh, detected_lines_map)
    
    # Clean small noise
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    text_only = cv2.morphologyEx(text_only, cv2.MORPH_OPEN, kernel_clean)
    
    cnts_text, _ = cv2.findContours(text_only, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    text_bottoms = []
    text_centers_y = []
    
    for c in cnts_text:
        if cv2.contourArea(c) < 15:
            continue
            
        tx, ty, tw, th = cv2.boundingRect(c)
        text_bottoms.append(ty + th)
        text_centers_y.append(ty + th/2)

    if not text_bottoms:
        return 1.0,copy_img

    # Use Median to determine text baseline position, ignoring outliers like descending letters ('ן', 'ך')
    median_text_bottom = np.median(text_bottoms)
    avg_text_y = np.mean(text_centers_y)

    best_line_y = 0
    min_dist_to_line = 99999
    
    # Identify which ruled line the text belongs to (closest line to text center)
    for line_c in valid_lines:
        lx, ly, lw, lh = cv2.boundingRect(line_c)
        current_line_y = ly 
        
        dist = abs(current_line_y - avg_text_y)
        if dist < min_dist_to_line:
            min_dist_to_line = dist
            best_line_y = current_line_y

    # Calculate distance: Positive = Above line, Negative = Below line
    distance = best_line_y - median_text_bottom
    
    # Sensitivity factor: determines the pixel range that maps to the 0-1 score
    SENSITIVITY = 60.0 
    
    # Normalize score: 0.5 represents text sitting exactly on the line
    raw_score = 0.5 + (distance / SENSITIVITY)
    
    # Clip result to strict 0.0 - 1.0 range
    final_score = max(0.0, min(1.0, raw_score))

    #visualization of the image
    cv2.line(copy_img,(0,int(best_line_y)),(img_w,int(best_line_y)),(255,0,0),2) #blue line for the printed baseline
    cv2.line(copy_img,(0,int(median_text_bottom)),(img_w,int(median_text_bottom)),(0,0,255),2) #red line for the bottom of the text
    cv2.rectangle(copy_img,(5,5),(225,50),(0,0,0),-1) #black rectangle as background for the greaded score
    cv2.putText(copy_img,f"Baseline: {final_score:.3f}",(10,30),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
    return final_score,copy_img

In [179]:
def process_images_for_position(
    input_dir: str,
    output_excel: str,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:
    # Main batch processing function: Iterates images, sorts them, and saves results to Excel
    
    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    # Sort using natural sort order (e.g., 1, 2, ... 10 instead of 1, 10, 2)
    image_files = sorted(list(image_files), key=lambda p: extract_file_id(p.name))
    
    total = len(image_files)

    print(f"Found {total} images to analyze")
    print(f"Output Excel file: {output_excel}")
    
    results = []

    for idx, img_path in enumerate(image_files, 1):
        try:
            file_id = extract_file_id(img_path.name)
            img = cv2.imread(str(img_path))

            if img is None:
                print(f"[WARNING] Could not read: {img_path.name}")
                results.append({
                    'ID_Number': file_id,
                    'Filename': img_path.name,
                    'Baseline': None
                })
                continue

            score,viz_img = calculate_position_score(img)
            
            # Fallback to neutral score (0.5) if detection fails
            if score is None:
                score = 0.5
                print(f"[INFO] Detection fallback for {img_path.name}")

            #saves image with results

            cv2.imwrite(os.path.join(visualization_folder,'baseline',img_path.name),viz_img)
            results.append({
                'ID_Number': file_id,
                'Filename': img_path.name,
                'Baseline': round(score, 3)
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")
            

        except Exception as e:
            print(f"[ERROR] Failed to process {img_path.name}: {str(e)}")
            results.append({
                'ID_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Baseline': None
            })

    print(f"Analysis complete!")

    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False, sheet_name='Line Position')

    print(f"\nExcel file saved: {output_excel}")
    print("\nPosition Statistics (0=Cutting, 0.5=On Line, 1=Floating):")
    print(f"Min: {df['Baseline'].min():.3f}")
    print(f"Max: {df['Baseline'].max():.3f}")
    print(f"Mean: {df['Baseline'].mean():.3f}")
    print(f"Median: {df['Baseline'].median():.3f}")
    print(f"Std Dev: {df['Baseline'].std():.3f}")

    return df


### Right Margin

In [180]:
def calculate_right_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    h, w = img_bgr.shape[:2]
    
    # זיהוי סוג הדף לפי שם הקובץ
    is_blank_page = "_COLUMNS" in filename
    
    if is_blank_page:
        # ---- לוגיקה לדף חלק (עמודות) ----
        b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
        red_mask = (r > 150) & (g < 80) & (b < 80)
        xs = np.where(red_mask.any(axis=0))[0]
        
        if len(xs) == 0:
            return -1.0 # לא נמצאו טורים
            
        # הקצה הימני ביותר של הטור הימני ביותר
        rightmost_x = int(xs[-1])
        
    else:
        # ---- לוגיקה לדף שורות----
        if len(img_bgr.shape) == 3:
            gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        else:
            gray = img_bgr.copy()
            
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        min_line_width = int(w * 0.4)
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
        detected_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
        text_only = cv2.subtract(thresh, detected_lines)
        
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(text_only, connectivity=8)
        clean_ink = np.zeros_like(text_only)
        for i in range(1, num_labels):
            x, y, bw, bh, area = stats[i]
            if area >= 15 and bh >= 8:
                clean_ink[labels == i] = 255
                
        search_width = max(1, int(w * 0.973))
        col_projection = np.sum(clean_ink[:, :search_width], axis=0)
        max_density = np.max(col_projection)
        
        if max_density == 0:
            return -1.0
            
        noise_threshold = max_density * 0.05
        ink_cols = np.where(col_projection > noise_threshold)[0]
        
        if len(ink_cols) == 0:
            return -1.0
            
        rightmost_x = int(np.max(ink_cols))

    # ---- חישוב הציון המשותף לשני סוגי הדפים ----
    # המרחק מנקודת הימין הקיצונית ועד סוף הדף
    right_margin_pixels = w - rightmost_x
    margin_ratio = right_margin_pixels / w
    
    # המרה לסקאלה המבוקשת: 0.0=צמוד לימין, 0.5=מאוזן (15%), 1.0=גדול מאוד (30%)
    MAX_MARGIN_RATIO = 0.30
    score = margin_ratio / MAX_MARGIN_RATIO
    grade = round(max(0.0, min(1.0, score)), 4)
    
    # ---- ציור ויזואליזציה אחידה ----
    if vis_dir and filename:
        vis = img_bgr.copy()
        overlay = vis.copy()
        
        # סימון השטח הריק מימין בירוק שקוף
        cv2.rectangle(overlay, (rightmost_x, 0), (w - 1, h - 1), (0, 200, 0), -1)
        vis = cv2.addWeighted(overlay, 0.25, vis, 0.75, 0)
        cv2.line(vis, (rightmost_x, 0), (rightmost_x, h), (0, 200, 0), 2)
        
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Right Margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        cv2.imwrite(os.path.join(vis_dir, filename), vis)
        
    return grade

In [181]:
def process_images_for_right_margin(
    input_dir: str,
    output_excel: str,
    vis_dir: str = None,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:

    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    image_files = sorted(list(image_files), key=lambda p: extract_file_id(p.name))
    total = len(image_files)

    print(f"Found {total} images to analyze for Right Margin")
    
    if vis_dir and not os.path.exists(vis_dir):
        os.makedirs(vis_dir)

    results = []

    for idx, img_path in enumerate(image_files, 1):
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # מעבירים את ה-filename ואת ה-vis_dir ישר לפונקציה כדי שתטפל בהכל
            grade = calculate_right_margin(img, filename=img_path.name, vis_dir=vis_dir)
            
            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Right_Margin': grade if grade != -1.0 else None
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")

        except Exception as e:
            print(f"[ERROR] {img_path.name}: {e}")
            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Right_Margin': None
            })

    # Save to Excel
    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False, sheet_name='Right Margin')

    # Inject Excel formula to extract numeric ID from filename
    wb = load_workbook(output_excel)
    ws = wb.active
    for row_num in range(2, len(results) + 2):
        ws[f'A{row_num}'] = (
            f'=VALUE(MID(B{row_num}, FIND("(",B{row_num})+1, '
            f'FIND(")",B{row_num})-FIND("(",B{row_num})-1))'
        )
    wb.save(output_excel)

    print(f"\nExcel file saved: {output_excel}")
    print("\nRight Margin Statistics (0=narrow, 1=wide):")
    print(f"Min:    {df['Right_Margin'].min():.3f}")
    print(f"Max:    {df['Right_Margin'].max():.3f}")
    print(f"Mean:   {df['Right_Margin'].mean():.3f}")
    
    return df

In [182]:
# process_images_for_right_margin(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, 'right_margin.xlsx'),
#     vis_dir=os.path.join(visualization_folder, 'right_margin')
# )

### Left Margin

In [183]:
def calculate_left_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    h, w = img_bgr.shape[:2]
    
    # זיהוי סוג הדף לפי שם הקובץ
    is_blank_page = "_COLUMNS" in filename
    
    if is_blank_page:
        # ---- לוגיקה לדף חלק (עמודות) ----
        b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
        red_mask = (r > 150) & (g < 80) & (b < 80)
        xs = np.where(red_mask.any(axis=0))[0]
        
        if len(xs) == 0:
            return -1.0 # לא נמצאו טורים
            
        # הקצה השמאלי ביותר של הטור השמאלי ביותר
        leftmost_x = int(xs[0])
        
    else:
        # ---- לוגיקה לדף שורות (הקוד המקורי שלך) ----
        if len(img_bgr.shape) == 3:
            gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        else:
            gray = img_bgr.copy()
            
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        min_line_width = int(w * 0.4)
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
        detected_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
        text_only = cv2.subtract(thresh, detected_lines)
        
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(text_only, connectivity=8)
        clean_ink = np.zeros_like(text_only)
        for i in range(1, num_labels):
            x, y, bw, bh, area = stats[i]
            if area >= 15 and bh >= 8:
                clean_ink[labels == i] = 255
                
        left_start = int(w * 0.027)
        col_projection = np.sum(clean_ink[:, left_start:], axis=0)
        max_density = np.max(col_projection)
        
        if max_density == 0:
            return -1.0
            
        noise_threshold = max_density * 0.05
        ink_cols = np.where(col_projection > noise_threshold)[0]
        
        if len(ink_cols) == 0:
            return -1.0
            
        leftmost_x = int(np.min(ink_cols)) + left_start

    # ---- חישוב הציון המשותף לשני סוגי הדפים ----
    # המרחק מתחילת הדף (משמאל) לנקודת הטקסט הראשונה
    margin_ratio = leftmost_x / w
    
    # המרה לסקאלה המבוקשת: 0.0=צמוד לשמאל, 0.5=מאוזן (15%), 1.0=גדול מאוד (30%)
    MAX_MARGIN_RATIO = 0.30
    score = margin_ratio / MAX_MARGIN_RATIO
    grade = round(max(0.0, min(1.0, score)), 4)
    
    # ---- ציור ויזואליזציה אחידה ----
    if vis_dir and filename:
        vis = img_bgr.copy()
        overlay = vis.copy()
        
        # סימון השטח הריק משמאל בירוק שקוף
        cv2.rectangle(overlay, (0, 0), (leftmost_x, h - 1), (0, 200, 0), -1)
        vis = cv2.addWeighted(overlay, 0.25, vis, 0.75, 0)
        cv2.line(vis, (leftmost_x, 0), (leftmost_x, h), (0, 200, 0), 2)
        
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Left Margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        cv2.imwrite(os.path.join(vis_dir, filename), vis)
        
    return grade

In [184]:
def process_images_for_left_margin(
    input_dir: str,
    output_excel: str,
    vis_dir: str = None,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:

    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    image_files = sorted(list(image_files), key=lambda p: extract_file_id(p.name))
    total = len(image_files)

    print(f"Found {total} images to analyze for Left Margin")
    
    if vis_dir and not os.path.exists(vis_dir):
        os.makedirs(vis_dir)

    results = []

    for idx, img_path in enumerate(image_files, 1):
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # מעבירים את השם והתיקייה כדי שתקרה ויזואליזציה (ויבדק סוג הדף)
            grade = calculate_left_margin(img, filename=img_path.name, vis_dir=vis_dir)
            
            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Left_Margin': grade if grade != -1.0 else None
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")

        except Exception as e:
            print(f"[ERROR] {img_path.name}: {e}")
            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Left_Margin': None
            })

    # Save to Excel
    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False, sheet_name='Left Margin')

    # Inject Excel formula to extract numeric ID from filename
    wb = load_workbook(output_excel)
    ws = wb.active
    for row_num in range(2, len(results) + 2):
        ws[f'A{row_num}'] = (
            f'=VALUE(MID(B{row_num}, FIND("(",B{row_num})+1, '
            f'FIND(")",B{row_num})-FIND("(",B{row_num})-1))'
        )
    wb.save(output_excel)

    print(f"\nExcel file saved: {output_excel}")
    print("\nLeft Margin Statistics (0=narrow, 1=wide):")
    print(f"Min:    {df['Left_Margin'].min():.3f}")
    print(f"Max:    {df['Left_Margin'].max():.3f}")
    print(f"Mean:   {df['Left_Margin'].mean():.3f}")
    
    return df

In [185]:
# process_images_for_left_margin(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, 'left_margin.xlsx'),
#     vis_dir=os.path.join(visualization_folder, 'left_margin')
# )

### Top Margin

In [186]:
def calculate_top_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    """Top margin = y of highest red rectangle border / image height."""
    h, w = img_bgr.shape[:2]
    b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
    
    # זיהוי הפיקסלים האדומים (היכן שיש טקסט מסומן מהשלב הקודם)
    red_mask = (r > 150) & (g < 80) & (b < 80)
    ys = np.where(red_mask.any(axis=1))[0]
    
    if len(ys) == 0:
        return -1.0
        
    # הפיקסל העליון ביותר שבו מתחיל הטקסט (השוליים העליונים)
    y_top = int(ys[0])
    
    # נרמול המרחק ביחס ל*גובה* התמונה (מכיוון שזהו ציר אנכי)
    margin_ratio = y_top / h
    
    # --- המרת הציון לפי הדרישה ---
    # 0.0 = צמוד לקצה העליון
    # 0.5 = מרחק מאוזן (מוגדר כ-15% מגובה הדף)
    # 1.0 = מרחק גדול מאוד (מוגדר כ-30% מגובה הדף ומעלה)
    MAX_MARGIN_RATIO = 0.30
    
    score = margin_ratio / MAX_MARGIN_RATIO
    grade = round(max(0.0, min(1.0, score)), 4)
    
    if vis_dir and filename:
        vis = img_bgr.copy()
        overlay = vis.copy()
        
        # סימון השטח הריק העליון בצהוב שקוף
        cv2.rectangle(overlay, (0, 0), (w, y_top), (0, 200, 200), -1)
        vis = cv2.addWeighted(overlay, 0.35, vis, 0.65, 0)
        
        # קו אדום שמסמן היכן מתחיל הטקסט
        cv2.line(vis, (0, y_top), (w, y_top), (0, 0, 255), 2)
        
        # הוספת הציון על גבי התמונה (עם רקע שחור לקריאות מקסימלית)
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Top Margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        cv2.imwrite(os.path.join(vis_dir, filename), vis)
        
    return grade

In [187]:
# process_images_for_top_margin(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, "top_margin.xlsx"),
#     vis_dir=os.path.join(visualization_folder, "top_margin"),
# )

### Bottom Margin

In [188]:
def calculate_bottom_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    """Bottom margin = (height - y of lowest red rectangle border) / image height."""
    h, w = img_bgr.shape[:2]
    b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
    red_mask = (r > 150) & (g < 80) & (b < 80)
    ys = np.where(red_mask.any(axis=1))[0]
    if len(ys) == 0:
        return -1
    y_bottom = int(ys[-1])
    grade = round(min(1.0, ((h - 1 - y_bottom) / h) / 0.30), 4)
    if vis_dir and filename:
        vis = img_bgr.copy()
        overlay = vis.copy()
        cv2.rectangle(overlay, (0, y_bottom), (w, h), (0, 200, 200), -1)
        vis = cv2.addWeighted(overlay, 0.35, vis, 0.65, 0)
        cv2.line(vis, (0, y_bottom), (w, y_bottom), (0, 0, 255), 2)
        cv2.putText(vis, f"bottom_margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)
    return grade


def process_images_for_bottom_margin(
        input_dir: str, output_excel: str,
        vis_dir: str = None, extensions: tuple = ('.png', '.jpg', '.jpeg')
) -> None:
    records = []
    images = sorted(p for p in Path(input_dir).iterdir()
                    if p.suffix.lower() in extensions and COLUMN_TOKEN in p.name)
    for img_path in images:
        img = cv2.imread(str(img_path))
        if img is None: continue
        vname = img_path.stem + "_bottom_margin.png" if vis_dir else ""
        grade = calculate_bottom_margin(img, filename=vname, vis_dir=vis_dir)
        records.append({"Filename": img_path.name, "Bottom_Margin": grade})
    pd.DataFrame(records).to_excel(output_excel, index=False)
    print(f"Saved bottom_margin.xlsx ({len(records)} rows)")

In [189]:
# process_images_for_bottom_margin(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, "bottom_margin.xlsx"),
#     vis_dir=os.path.join(visualization_folder, "bottom_margin"),
# )

### Column Spacing

In [190]:
def calculate_column_spacing(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    """Detect column bands from red rectangles, measure average gap between them."""
    h, w = img_bgr.shape[:2]
    b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
    
    # מזהה את המלבנים האדומים שהוגדרו בקובץ הנרמול
    red_mask = (r > 150) & (g < 80) & (b < 80)
    col_proj = red_mask.any(axis=0)
    
    bands, start = [], None
    for x, val in enumerate(col_proj):
        if val and start is None:
            start = x
        elif not val and start is not None:
            bands.append((start, x))
            start = None
    if start is not None:
        bands.append((start, w))
        
    if len(bands) < 2:
        return -1.0  # מחזיר -1 אם אי אפשר למדוד (פחות מ-2 טורים)
        
    # חישוב הרווחים (תחילת הטור הבא פחות סוף הטור הנוכחי)
    gaps = []
    for i in range(len(bands) - 1):
        gap = max(0, bands[i+1][0] - bands[i][1])
        gaps.append(gap)
        
    avg_gap = sum(gaps) / len(gaps)
    gap_ratio = avg_gap / w
    
    # --- המרת הציון לפי הדרישה שלך ---
    # 0.0 = צפוף/נוגע (טורים צמודים)
    # 0.5 = תקין (מוגדר כרגע כ-15% מרוחב הדף)
    # 1.0 = מרווח עצום (מוגדר כרגע כ-30% מהדף)
    MAX_GAP_RATIO = 0.30 
    
    score = gap_ratio / MAX_GAP_RATIO
    grade = round(max(0.0, min(1.0, score)), 4)
    
    # ציור ויזואליזציה (כדי שתוכל לראות את התוצאה בתיקיית visualization_results)
    if vis_dir and filename:
        vis = img_bgr.copy()
        for x1, x2 in bands:
            cv2.rectangle(vis, (x1, 0), (x2, h), (0, 200, 0), 2)
        for i in range(len(bands) - 1):
            gx1, gx2 = bands[i][1], bands[i+1][0]
            mid = h // 2
            cv2.rectangle(vis, (gx1, mid - 10), (gx2, mid + 10), (0, 0, 255), -1)
            cv2.putText(vis, f"{gaps[i]}px", (gx1 + 4, mid + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # הדפסת הציון הסופי על התמונה
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Column Spacing: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)
        
    return grade

In [191]:
# process_images_for_column_spacing(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, "column_spacing.xlsx"),
#     vis_dir=os.path.join(visualization_folder, "column_spacing"),
# )

### Word Spacing

### Letter Size 

## Word Aspect Ratio

In [192]:
def remove_printed_text_and_lines(img: np.ndarray) -> np.ndarray:
    # Preprocesses image to remove horizontal ruled lines and noise, isolating handwriting
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Create binary inverse (white text on black background)
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    height, width = img.shape[:2]

    # Detect and remove horizontal lines using Hough Transform
    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi/180, threshold=100, minLineLength=width*0.3, maxLineGap=10)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Draw thick lines over detected ruled lines to mask them
            cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 8)

    # Subtract lines from the binary image
    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))
    
    # Filter out noise and irrelevant regions using Connected Components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        # Define regions to ignore (header, footer) and noise threshold
        is_bottom_region = (y + h) > height * 0.85
        is_top_region = y < height * 0.1
        is_too_small = area < 15

        # Keep only valid handwriting components
        if not (is_bottom_region or is_top_region or is_too_small):
            handwriting_mask[labels == i] = 255

    # Invert back to normal (black text on white background)
    result = cv2.bitwise_not(handwriting_mask)
    return result

In [193]:
def calculate_dimension_raw(img: np.ndarray) -> float:
    # Calculates the raw Height/Width ratio using the median of handwriting blobs
    handwriting_only = remove_printed_text_and_lines(img)

    if len(handwriting_only.shape) == 3:
        gray = cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
    else:
        gray = handwriting_only

    # Binary thresholding (invert so text is white)
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    heights = []
    widths = []

    for c in contours:
        if cv2.contourArea(c) > 20:
            x, y, w, h = cv2.boundingRect(c)
            # Filter out tiny noise that might skew the median
            if w > 3 and h > 3:
                widths.append(w)
                heights.append(h)

    if not heights or not widths:
        return None

    median_h = np.median(heights)
    median_w = np.median(widths)

    if median_w == 0:
        return None

    # Calculate raw ratio
    raw_ratio = median_h / median_w

    return raw_ratio

## Basline Slope

In [194]:
USER_MIN_COMPONENT_AREA = 20
USER_MIN_COMPONENT_WIDTH = 3
USER_MIN_COMPONENT_HEIGHT = 4
USER_MIN_DENSITY = 0.18
USER_MIN_NEIGHBOR_SUPPORT = 2
USER_NEIGHBOR_RADIUS_X = 25
USER_NEIGHBOR_RADIUS_Y = 18


def remove_printed_text_and_lines(img: np.ndarray) -> np.ndarray:
    # Preprocesses image to remove horizontal ruled lines and noise, isolating handwriting
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    height, width = gray.shape[:2]

    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi/180, threshold=100, minLineLength=int(width * 0.30), maxLineGap=10)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if abs(y2 - y1) <= 3:
                cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 8)

    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))

    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary_no_lines = cv2.morphologyEx(binary_no_lines, cv2.MORPH_OPEN, kernel_open, iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        density = area / max(w * h, 1)
        is_bottom_region = (y + h) > height * 0.92
        is_top_region = y < height * 0.03
        is_too_small = area < USER_MIN_COMPONENT_AREA
        is_too_thin = w < USER_MIN_COMPONENT_WIDTH or h < USER_MIN_COMPONENT_HEIGHT
        is_too_sparse = density < USER_MIN_DENSITY

        if not (is_bottom_region or is_top_region or is_too_small or is_too_thin or is_too_sparse):
            handwriting_mask[labels == i] = 255

    result = cv2.bitwise_not(handwriting_mask)
    return result


def extract_file_id(filename: str) -> int:
    match = re.search(r'(\d+)', filename)
    if match:
        return int(match.group(1))
    return -1


def get_letter_centroids(img: np.ndarray):
    # Extracts letter-like connected components and keeps only concentrated pixels with local support
    handwriting_only = remove_printed_text_and_lines(img)

    if len(handwriting_only.shape) == 3:
        gray = cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
    else:
        gray = handwriting_only

    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    height, width = binary.shape[:2]

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

    candidates = []
    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        cx, cy = centroids[i]
        density = area / max(w * h, 1)

        if area < USER_MIN_COMPONENT_AREA:
            continue
        if w < USER_MIN_COMPONENT_WIDTH or h < USER_MIN_COMPONENT_HEIGHT:
            continue
        if density < USER_MIN_DENSITY:
            continue
        if y < height * 0.03 or (y + h) > height * 0.92:
            continue

        candidates.append({
            'x': x,
            'y': y,
            'w': w,
            'h': h,
            'cx': float(cx),
            'cy': float(cy),
            'area': area,
            'density': float(density)
        })

    letters = []
    for i, comp in enumerate(candidates):
        neighbors = 0
        for j, other in enumerate(candidates):
            if i == j:
                continue
            if abs(comp['cx'] - other['cx']) <= USER_NEIGHBOR_RADIUS_X and abs(comp['cy'] - other['cy']) <= USER_NEIGHBOR_RADIUS_Y:
                neighbors += 1
        if neighbors >= USER_MIN_NEIGHBOR_SUPPORT:
            letters.append(comp)

    letters = sorted(letters, key=lambda d: d['cx'], reverse=True)
    return letters


def calculate_baseline_slope_raw(img: np.ndarray) -> float:
    # Calculates raw line flow direction using linear regression on concentrated letter centroids (RTL aware)
    letters = get_letter_centroids(img)

    if len(letters) < 2:
        return None

    xs = np.array([l['cx'] for l in letters], dtype=np.float32)
    ys = np.array([l['cy'] for l in letters], dtype=np.float32)

    try:
        m, b = np.polyfit(xs, ys, 1)
    except Exception:
        return None

    right_x = np.max(xs)
    left_x = np.min(xs)
    right_y = m * right_x + b
    left_y = m * left_x + b

    raw_diff = right_y - left_y
    return float(raw_diff)

# Roundness vs. Angularity

In [195]:
USER_MIN_PERCENTILE = 5
USER_MAX_PERCENTILE = 95

# קרנל "תפירה" א-סימטרי: 6 לרוחב, 8 לגובה. 
# הגובה הגדול יותר נועד "לרפא" את החתך האופקי שנוצר בעקבות מחיקת השורה המודפסת.
USER_CLOSING_KERNEL_SIZE = (6, 8) 

USER_MIN_HOLE_AREA = 7
# ==========================================

def extract_file_id(filename: str) -> int:
    match = re.search(r'(\d+)', filename)
    if match:
        return int(match.group(1))
    return -1

def remove_printed_text_and_lines(img: np.ndarray) -> np.ndarray:
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # OTSU לשמירה על שלמות הדיו של הכותב
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    height, width = img.shape[:2]

    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi/180, threshold=100, minLineLength=width*0.3, maxLineGap=10)
    
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # === השינוי: מחיקה כירורגית דקה (3 פיקסלים) במקום 8 עבים ===
            cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 3)

    # החסרת הקווים
    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))
    
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        is_bottom_region = (y + h) > height * 0.85
        is_top_region = y < height * 0.1
        is_too_small = area < 15

        if not (is_bottom_region or is_top_region or is_too_small):
            handwriting_mask[labels == i] = 255

    result = cv2.bitwise_not(handwriting_mask)
    return result

In [196]:
def calculate_roundness_raw(img: np.ndarray, filename: str, vis_dir: str = None) -> float:
    # 1. הפעלת מחיקת השורות המודפסות
    handwriting_only = remove_printed_text_and_lines(img)

    if len(handwriting_only.shape) == 3:
        gray = cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
    else:
        gray = handwriting_only

    # המרה לבינארי שוב אחרי הניקוי
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 2. איחוי ותפירת חתכים (כולל החתך שהשארנו ממחיקת השורה)
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, USER_CLOSING_KERNEL_SIZE)
    closed_bin = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, close_kernel)

    # 3. מציאת קווי מתאר והיררכיה (לאיתור חללים)
    contours, hierarchy = cv2.findContours(closed_bin, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

    if hierarchy is None:
        return 0.500

    circularities = []
    debug_img = None
    if vis_dir:
        debug_img = img.copy() if len(img.shape) == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    for i, c in enumerate(contours):
        parent_idx = hierarchy[0][i][3]
        
        # זיהוי חור
        if parent_idx != -1: 
            area = cv2.contourArea(c)
            # סינון רעשים ואזורים גדולים מדי
            if USER_MIN_HOLE_AREA < area < 3000:
                perimeter = cv2.arcLength(c, True)
                
                if perimeter > 0:
                    circularity = (4 * np.pi * area) / (perimeter ** 2)
                    circularity = min(1.0, circularity)
                    circularities.append(circularity)

                    if debug_img is not None:
                        green = int(circularity * 255)
                        red = int((1 - circularity) * 255)
                        cv2.drawContours(debug_img, [c], -1, (0, green, red), 2)

    if not circularities:
        if vis_dir and debug_img is not None:
            cv2.putText(debug_img, "No Holes Found. Assigned Default 0.500", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.imwrite(os.path.join(vis_dir, f"{filename}"), debug_img)
        return 0.500

    median_circularity = np.median(circularities)

    if vis_dir and debug_img is not None:
        cv2.putText(debug_img, f"Raw Roundness: {median_circularity:.3f}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
        cv2.imwrite(os.path.join(vis_dir, f"{filename}"), debug_img)

    return median_circularity


## Run All Feature Extraction

In [197]:
# Per-image grade functions — same cap-based pattern as top/bottom/column spacing.
# No dataset stats needed; each image grades itself 0-1.

def grade_thickness(img):
    """Stroke width as fraction of image height, capped at 8%."""
    raw = calculate_stroke_thickness_pure(img)
    if raw is None or raw == 0:
        return None
    return round(min(1.0, (raw / img.shape[0]) / 0.08), 3)

def grade_aspect_ratio(img):
    """Median letter H/W ratio, capped at 2.5 (very tall/narrow letters)."""
    raw = calculate_dimension_raw(img)
    if raw is None:
        return None
    return round(min(1.0, raw / 2.5), 3)

def grade_slope(img):
    """Baseline slope: 0.5=flat, <0.5=ascending, >0.5=descending.
    Raw pixel diff normalised by image width, capped at ±15% of width."""
    raw = calculate_baseline_slope_raw(img)
    if raw is None:
        return 0.500
    return round(max(0.0, min(1.0, 0.5 + (raw / img.shape[1]) / 0.30)), 3)

def grade_roundness(img, fname, vis_dir):
    """Circularity of letter holes — already 0-1 by definition, just clamped."""
    raw = calculate_roundness_raw(img, fname, vis_dir)
    if raw is None:
        return None
    if raw == 0.500:
        return 0.500
    return round(max(0.0, min(1.0, float(raw))), 3)


In [198]:
import glob
import numpy as np

images = sorted(
    glob.glob(os.path.join(normalised_images_folder, "*.png")) +
    glob.glob(os.path.join(normalised_images_folder, "*.jpg"))
)

blank_records = []
line_records  = []

vis_baseline  = os.path.join(visualization_folder, "baseline")
vis_thickness = os.path.join(visualization_folder, "stroke_thickness")
vis_aspect    = os.path.join(visualization_folder, "word_aspect_ratio")
vis_slope     = os.path.join(visualization_folder, "baseline_slope")
vis_roundness = os.path.join(visualization_folder, "roundness")

for d in [vis_baseline, vis_thickness, vis_aspect, vis_slope, vis_roundness]:
    os.makedirs(d, exist_ok=True)

def _ov(im, text):
    out = im.copy()
    cv2.rectangle(out, (5, 5), (370, 50), (0, 0, 0), -1)
    cv2.putText(out, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    return out

def _safe(fn, default=None):
    try:
        return fn()
    except Exception as e:
        print(f"    [warn] {e}")
        return default

# ── Loop: grade every image ──────────────────────────────────────────────────
for img_path in images:
    fname = os.path.basename(img_path)
    img   = cv2.imread(img_path)
    print(f"Processing: {fname}")

    if COLUMN_TOKEN in fname:
        top     = calculate_top_margin(img,
                      filename=fname.replace(".png", "_top_margin.png"),
                      vis_dir=os.path.join(visualization_folder, "top_margin"))
        bottom  = calculate_bottom_margin(img,
                      filename=fname.replace(".png", "_bottom_margin.png"),
                      vis_dir=os.path.join(visualization_folder, "bottom_margin"))
        spacing = calculate_column_spacing(img,
                      filename=fname.replace(".png", "_col_spacing.png"),
                      vis_dir=os.path.join(visualization_folder, "column_spacing"))
        print(f"  [blank] top_margin     = {top}")
        print(f"  [blank] bottom_margin  = {bottom}")
        print(f"  [blank] col_spacing    = {spacing}")
        blank_records.append({
            "Filename":       fname,
            "Top_Margin":     top,
            "Bottom_Margin":  bottom,
            "Column_Spacing": spacing,
        })

    else:
        slant_raw    = _safe(lambda: extract_slant(img_path))
        slant_grade  = round(float(slant_raw[0]), 3) if slant_raw is not None else None

        thickness_grade = _safe(lambda: grade_thickness(img))

        pos_result   = _safe(lambda: calculate_position_score(img), default=(None, None))
        if isinstance(pos_result, tuple):
            baseline_score, vis_base = pos_result
        else:
            baseline_score, vis_base = pos_result, None
        baseline_grade = round(float(baseline_score), 3) if baseline_score is not None else None
        if vis_base is not None:
            cv2.imwrite(os.path.join(vis_baseline, fname), vis_base)

        aspect_grade   = _safe(lambda: grade_aspect_ratio(img))
        slope_grade    = _safe(lambda: grade_slope(img), default=0.500)
        roundness_grade = _safe(lambda: grade_roundness(img, fname, vis_roundness))

        print(f"  [line] slant            = {slant_grade}")
        print(f"  [line] stroke_thickness = {thickness_grade}")
        print(f"  [line] baseline         = {baseline_grade}")
        print(f"  [line] word_aspect_ratio= {aspect_grade}")
        print(f"  [line] baseline_slope   = {slope_grade}")
        print(f"  [line] roundness        = {roundness_grade}")
        print("  [line] word_spacing     = (run separately)")
        print("  [line] letter_size      = (run separately)")

        if thickness_grade is not None:
            cv2.imwrite(os.path.join(vis_thickness, fname),
                        _ov(img, f"Stroke Thickness: {thickness_grade:.3f}"))
        if aspect_grade is not None:
            cv2.imwrite(os.path.join(vis_aspect, fname),
                        _ov(img, f"Word Aspect Ratio: {aspect_grade:.3f}"))
        if slope_grade is not None:
            slope_img = img.copy()
            cv2.rectangle(slope_img, (5, 5), (370, 50), (0, 0, 0), -1)
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
            letters  = get_letter_centroids(img_gray)
            if len(letters) >= 2:
                xs = np.array([l["cx"] for l in letters], dtype=np.float32)
                ys = np.array([l["cy"] for l in letters], dtype=np.float32)
                m_fit, b_fit = np.polyfit(xs, ys, 1)
                cv2.line(slope_img,
                         (int(np.min(xs)), int(m_fit * np.min(xs) + b_fit)),
                         (int(np.max(xs)), int(m_fit * np.max(xs) + b_fit)),
                         (0, 0, 255), 2)
                for l in letters:
                    cv2.circle(slope_img, (int(l["cx"]), int(l["cy"])), 2, (255, 0, 0), -1)
            cv2.putText(slope_img, f"Baseline Slope: {slope_grade:.3f}",
                        (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.imwrite(os.path.join(vis_slope, fname), slope_img)
        if roundness_grade is not None:
            rnd_path = os.path.join(vis_roundness, fname)
            if os.path.exists(rnd_path):
                rnd_img = cv2.imread(rnd_path)
                if rnd_img is not None:
                    cv2.rectangle(rnd_img, (5, 5), (370, 50), (0, 0, 0), -1)
                    cv2.putText(rnd_img, f"Roundness: {roundness_grade:.3f}",
                                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                    cv2.imwrite(rnd_path, rnd_img)

        line_records.append({
            "Filename":          fname,
            "Slant":             slant_grade,
            "Stroke_Thickness":  thickness_grade,
            "Baseline":          baseline_grade,
            "Word_Aspect_Ratio": aspect_grade,
            "Baseline_Slope":    slope_grade,
            "Roundness":         roundness_grade,
        })

    print("  [shared] left_margin    = (run separately)")
    print("  [shared] right_margin   = (run separately)")
    print()

# ── Save line features to Excel ──────────────────────────────────────────────
if line_records:
    df_line = pd.DataFrame(line_records).fillna(-1)
    excel_map = {
        "Slant":             "slant.xlsx",
        "Stroke_Thickness":  "stroke_thickness.xlsx",
        "Baseline":          "baseline.xlsx",
        "Word_Aspect_Ratio": "word_aspect_ratio.xlsx",
        "Baseline_Slope":    "baseline_slope.xlsx",
        "Roundness":         "roundness.xlsx",
    }
    for col, xfile in excel_map.items():
        out_path = os.path.join(feature_tables_location, xfile)
        df_line[["Filename", col]].to_excel(out_path, index=False)
        print(f"Saved {xfile} ({len(df_line)} rows)")
    print(f"\nline_features complete: {len(df_line)} images")

# ── Save blank page results to Excel ─────────────────────────────────────────
if blank_records:
    df_blank = pd.DataFrame(blank_records).fillna(-1)
    df_blank[["Filename", "Top_Margin"]].to_excel(
        os.path.join(feature_tables_location, "top_margin.xlsx"), index=False)
    df_blank[["Filename", "Bottom_Margin"]].to_excel(
        os.path.join(feature_tables_location, "bottom_margin.xlsx"), index=False)
    df_blank[["Filename", "Column_Spacing"]].to_excel(
        os.path.join(feature_tables_location, "column_spacing.xlsx"), index=False)
    print(f"Saved top_margin, bottom_margin, column_spacing tables ({len(blank_records)} rows)")


Processing: sd_COLUMNS.png
  [blank] top_margin     = 0.151
  [blank] bottom_margin  = 0.4769
  [blank] col_spacing    = 0.5099
  [shared] left_margin    = (run separately)
  [shared] right_margin   = (run separately)

Processing: sd_line_01.png
  [line] slant            = 0.502
  [line] stroke_thickness = 0.257999986410141
  [line] baseline         = 0.825
  [line] word_aspect_ratio= 0.837
  [line] baseline_slope   = 0.5
  [line] roundness        = 0.758
  [line] word_spacing     = (run separately)
  [line] letter_size      = (run separately)
  [shared] left_margin    = (run separately)
  [shared] right_margin   = (run separately)

Processing: sd_line_02.png
  [line] slant            = 0.512
  [line] stroke_thickness = 0.2630000114440918
  [line] baseline         = 0.95
  [line] word_aspect_ratio= 0.7
  [line] baseline_slope   = 0.5
  [line] roundness        = 0.822
  [line] word_spacing     = (run separately)
  [line] letter_size      = (run separately)
  [shared] left_margin    = (r

## Unified Feature Table

Merge all per-feature Excel files into a single table — one row per image, one column per feature.

## Graphological Report

Send the unified feature table to Claude Opus and receive a full personal graphological report.